In [1]:
using SpeedyWeather

In [2]:
spectral_grid = SpectralGrid()

output = NetCDFOutput(spectral_grid)
add!(output, SpeedyWeather.SurfaceShortwaveDownOutput())

NetCDFOutput{Field{Float32, 1, Vector{Float32}, FullGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ status: inactive/uninitialized
├ write restart file: true (if active)
├ interpolator: AnvilInterpolator{Float32, RingGrids.GridGeometry{OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Vector{Float32}, Vector{Int64}}, RingGrids.AnvilLocator{Float32, Vector{Float32}, Vector{Int64}}}
├ path: output.nc (overwrite=false)
├ frequency: 21600 seconds
└┐ variables:
 ├ v: meridional wind [m/s]
 ├ u: zonal wind [m/s]
 ├ srd: Surface shortwave radiation down [W/m^2]
 └ vor: relative vorticity [s^-1]

In [3]:
my_planet = Earth(spectral_grid, axial_tilt = -40)
my_planet2 = Earth(spectral_grid, axial_tilt = 40)

# Define and initialise the model
model = PrimitiveWetModel(
    spectral_grid;
    output = output,
    planet = my_planet,
    shortwave_radiation = TransparentShortwave()
)

model2 = PrimitiveWetModel(
    spectral_grid;
    output = output,
    planet = my_planet2,
    shortwave_radiation = TransparentShortwave()
)


simulation = initialize!(model)
simulation2 = initialize!(model2)

Simulation{PrimitiveWetModel}
├ prognostic_variables::PrognosticVariables{...}
├ diagnostic_variables::DiagnosticVariables{...}
└ model::PrimitiveWetModel{...}

In [4]:

# my_planet = Earth(spectral_grid, axial_tilt = -40)
# my_planet2 = Earth(spectral_grid, axial_tilt = 40)

# # Define and initialise the model
# model = PrimitiveWetModel(spectral_grid; planet=my_planet)
# model2 = PrimitiveWetModel(spectral_grid; planet=my_planet2)

# simulation = initialize!(model)
# simulation2 = initialize!(model2)

In [5]:
run!(simulation, period=Day(150), output=true)
run!(simulation2, period=Day(150), output=true)

In [7]:
using GLMakie
flux2 = simulation2.diagnostic_variables.physics.surface_shortwave_down
GLMakie.heatmap(flux2, title="Incoming solar flux")